In [1]:
# libraries:
import numpy as np
import torch

import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))

from autoencoder import SINDyConvAE
from training import train_sindy_convae, save_model_and_params
from sindy_utils import library_size

from example_dg import get_dg_data, get_dg_loaders


[INFO] Global seed set to 42 (PyTorch, NumPy, Random)


In [2]:
# physical parameters
amplitude = 0.25
frequency = 5.0     

# spatial discretization
nx = 64
ny = 32
Lx = 2.0
Ly = 1.0

# temporal discretization
dt = 0.05
T = 100.0

# Dataset creation
train_d, val_d, test_d = get_dg_data(randomized=False,
                                     amplitude=amplitude, frequency=frequency, 
                                     Lx=Lx, Ly=Ly, nx=nx, ny=ny,
                                     T=T, dt=dt)



In [ ]:
# ConvAE+SINDy parameters
params={}

# CAE structure
params['input_shape'] = (nx, ny, 2)
params['filter_widths'] = [16, 32]

# SINDy latent library
params['latent_dim'] = 1
params['poly_order'] = 5
params['include_sine'] = True
params['library_dim'] = library_size(params['latent_dim'], params['poly_order'], params['include_sine'])

# Loss penalties
params['loss_weights'] = {
    'decoder': 1.0,                     # Reconstruction x
    'sindy_x': 1e-3,                    # Reconstruction dx 
    'sindy_z': 0, # 1e-3,               # SINDy latent  dz/dt = Theta * Xi
    'sindy_reg': 1e-6                   # Sparsity promoter (L1)
}

# Thresholding
params['sequential_thresholding'] = True
params['threshold_frequency'] = 100
params['coefficient_threshold'] = 1e-2

# Training
params['activation'] = 'tanh'
params['learning_rate'] = 1e-3
params['batch_size'] = 32
params['print_frequency'] = 20

params['max_epochs'] = 201
params['refinement_epochs'] = 201


In [4]:
# Torch loaders
train_loader, val_loader, test_loader = get_dg_loaders(train_d, val_d, test_d, 
                                                       batch_size=params['batch_size'],
                                                       channels_first=False,
                                                       train_shuffle=True,
                                                       include_time=False)

In [5]:
# Model
model = SINDyConvAE(params)

# Train of the model
history = train_sindy_convae(model, train_loader, val_loader, params)

# Save
save_model_and_params(model, params, folder_path="./saved", model_name="model_partial")



[INFO] SINDy Autoencoder Configuration:
  > Decoder weight:  1.00e+00
  > SINDy x weight:  1.00e-03
  > SINDy z weight:  0.00e+00
  > SINDy Reg (L1):  1.00e-06
[INFO] Computation norms on validation set...
[INFO] Validation X norm (MSE baseline): 2.4992e-02
[INFO] Validation dX norm (MSE baseline): 1.1170e-01

--- PHASE 1: TRAINING ---
[TRAIN] Epoch    0
  Train Loss: 3.2272e-02 ( Dec: 3.22e-02 | dX: 1.11e-01 | dZ: 4.86e-02 | Reg: 5.65e-01 )
  Valid Loss: 1.4161e-02 ( Dec: 1.40e-02 | dX: 1.15e-01 | dZ: 1.05e-01 )
  >> RATIO: Decoder: 0.5620 | SINDy: 1.0261
[TRAIN] Epoch   20
  Train Loss: 8.4178e-05 ( Dec: 8.07e-05 | dX: 3.43e-03 | dZ: 6.50e+00 | Reg: 4.70e-02 )
  Valid Loss: 7.9246e-05 ( Dec: 7.59e-05 | dX: 3.29e-03 | dZ: 6.61e+00 )
  >> RATIO: Decoder: 0.0030 | SINDy: 0.0294
[TRAIN] Epoch   40
  Train Loss: 2.5749e-05 ( Dec: 2.48e-05 | dX: 9.18e-04 | dZ: 6.88e+00 | Reg: 1.52e-04 )
  Valid Loss: 2.4547e-05 ( Dec: 2.37e-05 | dX: 8.94e-04 | dZ: 7.02e+00 )
  >> RATIO: Decoder: 0.0009 | 

In [6]:
# Results analysis
with torch.no_grad():
    # .detach() removes the tensor from the trainable objects
    final_xi = (model.sindy_coeffs * model.sindy_mask).detach().cpu().numpy()

print("Xi final:")
print(np.round(final_xi, 4)) 

active_mask = np.abs(final_xi) > 0
print(f"Active coefficients: {np.sum(active_mask)}")

Xi final:
[[-0.]
 [-0.]
 [ 0.]
 [ 0.]
 [ 0.]
 [-0.]
 [ 0.]]
Active coefficients: 0


In [7]:
# Torch loaders with train not shuffled and time data
train_loader_new, _, _ = get_dg_loaders(train_d, val_d, test_d,
                                        batch_size=params['batch_size'],
                                        channels_first=False,
                                        train_shuffle=False, 
                                        include_time=True)

In [8]:
import matplotlib.pyplot as plt

def evaluate_and_save_zt(model, loader, device='cpu'):
    model.eval()
    z_list = []
    t_list = []
    
    model.to(device)
    
    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(device) # x = first element
            t = batch[2].cpu().numpy() # t = third element
            
            # encode the data
            z = model.encode(x)
            
            # gather z and t
            z_list.append(z.cpu().numpy())
            t_list.append(t)
            
    # concatenation and flattening
    z_all = np.concatenate(z_list).flatten()
    t_all = np.concatenate(t_list).flatten()
    
    # plot
    # plt.figure(figsize=(12, 5))
    # plt.plot(t_all, z_all, color='darkorange', linewidth=2, label='Latent Evolution $z(t)$')
    # plt.xlabel('Time $t$ (from loader)')
    # plt.ylabel('Latent Variable $z$')
    # plt.title('Latent trajecotry against time')
    # plt.grid(True, linestyle='--', alpha=0.6)
    # plt.legend()
    # plt.show()
    
    # latent data
    zt_array = np.column_stack((z_all, t_all)) # shape (N,2)
    
    return zt_array


zt_data = evaluate_and_save_zt(model, train_loader_new, device='cpu')

np.save('saved/latent_training_data.npy', zt_data)